In [1]:
import cuml

from cuml.ensemble import RandomForestRegressor

from sklearn import ensemble
from sklearn.ensemble import ExtraTreesRegressor, ExtraTreesClassifier

import sklearn

import matplotlib.pyplot as plt

import xgboost as xgb
from lightgbm import LGBMRegressor, LGBMClassifier
import lightgbm as lgb

import torch

import numpy as np
import cupy as cp

import pandas as pd
from pathlib import Path
import gc

from sklearn.metrics import r2_score

import time

In [2]:
# prepare data

datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/shadows-detection-dataset/trees-gt-shadows.csv")

#making sure
datasetPath.is_file()

True

In [3]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,180.0,-71.945084,19.489212,0.0,-19.35,-0.0,0.26,0.0


In [4]:
target_col = dataset["grviOut"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0.26, 0.26, 0.26, ..., 0.26, 0.26, 0.26],
      shape=(332640,), dtype=float32)

In [5]:
inputs = dataset.drop("grviOut", axis=1)
inputs = inputs.drop("sunRoll", axis=1)
inputs = inputs.drop("camRoll", axis=1)

inputs.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunPitch,sunYaw,camPitch,camYaw,shadows
0,0.248512,0.000147,0.000204,1136.471,1031.228,0.0,-71.945084,19.489212,-19.35,-0.0,0.0
1,0.209356,0.000254,0.000347,1136.471,1031.228,1.0,-71.945084,19.489212,-19.35,-0.0,0.0
2,0.228622,0.000229,0.000317,1136.471,1031.228,2.0,-71.945084,19.489212,-19.35,-0.0,0.0
3,0.193133,0.000223,0.000293,1136.471,1031.228,3.0,-71.945084,19.489212,-19.35,-0.0,0.0
4,0.258888,0.000187,0.000269,1136.471,1031.228,4.0,-71.945084,19.489212,-19.35,-0.0,0.0


In [6]:
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 2.4851234e-01,  1.4691126e-04,  2.0387112e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       [ 2.0935635e-01,  2.5367978e-04,  3.4656093e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       [ 2.2862221e-01,  2.2864803e-04,  3.1710259e-04, ...,
        -1.9350000e+01, -0.0000000e+00,  0.0000000e+00],
       ...,
       [ 8.0871526e-03,  3.2670033e-01,  3.1690043e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00],
       [ 1.2527064e-02,  2.7300212e-01,  2.6233262e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00],
       [ 2.2494195e-02,  2.6528352e-01,  2.5932705e-01, ...,
        -2.2218000e+01, -1.8000000e+02,  0.0000000e+00]],
      shape=(332640, 11), dtype=float32)

In [7]:
n_total = len(inputs)

n_train = int(0.9*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]

In [8]:
mask_high = val_inputs[:, 10] == 0.0
mask_low = ~mask_high

val_inputs_high = val_inputs[mask_high]
val_labels_high = val_labels[mask_high]

val_inputs_low = val_inputs[mask_low]
val_labels_low = val_labels[mask_low]

val_inputs = np.delete(val_inputs, 10, axis=1)
val_inputs_high = np.delete(val_inputs_high, 10, axis=1)
val_inputs_low = np.delete(val_inputs_low, 10, axis=1)

In [9]:
mask_high = train_inputs[:, 10] == 0.0
mask_low = ~mask_high

train_inputs_high = train_inputs[mask_high]
train_labels_high = train_labels[mask_high]

train_inputs_low = train_inputs[mask_low]
train_labels_low = train_labels[mask_low]

train_inputs = np.delete(train_inputs, 10, axis=1)
train_inputs_high = np.delete(train_inputs_high, 10, axis=1)
train_inputs_low = np.delete(train_inputs_low, 10, axis=1)

In [10]:
modelXGBlow = xgb.XGBRegressor(n_estimators = 500, max_depth = 15, min_child_weight = 3)
modelXGBhigh = xgb.XGBRegressor(n_estimators = 500, max_depth = 15, min_child_weight = 3)

In [11]:
# train shadows model

modelXGBlow = modelXGBlow.fit(train_inputs_low, train_labels_low)
modelXGBlow.save_model("xgb_low.json")

y_pred = modelXGBlow.predict(val_inputs_low)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_low, y_pred))
r2 = r2_score(val_labels_low, y_pred)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())

print ("Low RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2: " + str(r2))

Low RelRMSE: 3.064734773149369%, r2: 0.9761309027671814


In [12]:
# train clear model

modelXGBhigh = modelXGBhigh.fit(train_inputs_high, train_labels_high)
modelXGBlow.save_model("xgb_high.json")

y_pred = modelXGBhigh.predict(val_inputs_high)

rmse = cp.sqrt(cuml.metrics.mean_squared_error(val_labels_high, y_pred))
r2 = r2_score(val_labels_high, y_pred)

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())

print ("High RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2: " + str(r2))

High RelRMSE: 5.377299370404392%, r2: 0.9362559914588928


In [13]:
shadowsClass = xgb.XGBClassifier()
shadowsClass.load_model("best_xgb.json")

XGBoostError: [00:20:23] ../src/common/io.cc:102: Opening best_xgb.json failed: No such file or directory
Stack trace:
  [bt] (0) /home/philipm/miniconda3/envs/hls4ml-tutorial/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x194383) [0x72b971794383]
  [bt] (1) /home/philipm/miniconda3/envs/hls4ml-tutorial/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x194ae3) [0x72b971794ae3]
  [bt] (2) /home/philipm/miniconda3/envs/hls4ml-tutorial/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(+0x144770) [0x72b971744770]
  [bt] (3) /home/philipm/miniconda3/envs/hls4ml-tutorial/lib/python3.10/site-packages/xgboost/lib/libxgboost.so(XGBoosterLoadModel+0x185) [0x72b971744b55]
  [bt] (4) /home/philipm/miniconda3/envs/hls4ml-tutorial/lib/python3.10/lib-dynload/../../libffi.so.8(+0x6d8a) [0x72bb8506dd8a]
  [bt] (5) /home/philipm/miniconda3/envs/hls4ml-tutorial/lib/python3.10/lib-dynload/../../libffi.so.8(+0x61cd) [0x72bb8506d1cd]
  [bt] (6) /home/philipm/miniconda3/envs/hls4ml-tu

In [ ]:
results = []

for sample in val_inputs:
    if (round(shadowsClass.predict(np.array(sample).reshape(1, -1)).item())):
        results.append(modelXGBlow.predict(np.array(sample).reshape(1, -1)).item())
    else:
        results.append(modelXGBhigh.predict(np.array(sample).reshape(1, -1)).item())


rmse = np.sqrt(cuml.metrics.mean_squared_error(np.array(val_labels).reshape(-1), np.array(results, dtype=float).reshape(-1)))
r2 = r2_score(val_labels, np.array(results, dtype=float).reshape(-1))

rel_test_loss = (rmse*100) / (val_labels.max()-val_labels.min())

print ("RelRMSE: " + str(float(rel_test_loss)) + "%, " + "r2: " + str(r2))

In [ ]:
import time

In [ ]:
from cuml import ForestInference


inference_times_CPU = []
for i in range(0, 100):
    start_time = time.time()
    
    pred = round(shadowsClass.predict(np.array(sample).reshape(1, -1)).item())
    
    if (pred):
        results.append(modelXGBlow.predict(np.array(sample).reshape(1, -1)).item())
    else:
        results.append(modelXGBhigh.predict(np.array(sample).reshape(1, -1)).item())

    end_time = time.time()

    inference_times_CPU.append(end_time - start_time)

inference_times_CPU = np.array(inference_times_CPU, dtype = "float32")
print (f"CPU time: {inference_times_CPU.mean()} ({inference_times_CPU.std()})")


fil_model_XGB_low = ForestInference.load(
    "xgb_low.json",
    model_type="xgboost_json",
    is_classifier=False,
    output_type="cupy"
)

fil_model_XGB_high = ForestInference.load(
    "xgb_high.json",
    model_type="xgboost_json",
    is_classifier=False,
    output_type="cupy"
)

fil_shadowsClass = ForestInference.load(
    "best_xgb.json",
    model_type="xgboost_json",
    is_classifier=True,
    output_type="cupy"
)

with cuml.using_output_type("cupy"):

    for i in range(0, 20):
        x_gpu = cp.asarray(val_inputs[i:i+1], dtype=cp.float32)
        pred = fil_model_XGB_low.predict(x_gpu)
        pred_cpu = cp.asnumpy(pred)


    inference_times_GPU = []
    cputogpu = []
    gputocpu = []

    overall = []
    for i in range(0, 100):
        cp.cuda.Device().synchronize()
        general_start = time.time()
        start_time = time.time()

        x_gpu = cp.asarray(val_inputs[i:i+1], dtype=cp.float32)

        cp.cuda.Device().synchronize()
        end_time = time.time()
        
        t1 = end_time - start_time

        #########################################################

        cp.cuda.Device().synchronize()
        start_time = time.time()
        
        pred = cp.round(fil_shadowsClass.predict(x_gpu))
        
        if (pred):
            pred = fil_model_XGB_low.predict(x_gpu)
        else:
            pred = fil_model_XGB_high.predict(x_gpu)

        cp.cuda.Device().synchronize()
        end_time = time.time()

        t2 = end_time - start_time

        #########################################################

        cp.cuda.Device().synchronize()
        start_time = time.time()
        
        pred_cpu = cp.asnumpy(pred)

        cp.cuda.Device().synchronize()
        general_end = time.time()
        end_time = time.time()

        t3 = end_time - start_time

        #########################################################      

        cputogpu.append(t1)
        inference_times_GPU.append(t2)
        gputocpu.append(t3)

        overall.append(general_end - general_start)

    cputogpu = np.array(cputogpu, dtype = "float32")
    inference_times_GPU = np.array(inference_times_GPU, dtype = "float32")
    gputospu = np.array(gputocpu, dtype = "float32")

    overall = np.array(overall, dtype = "float32")
    
print(f"CPU to GPU time: {cputogpu.mean()} ({cputogpu.std()})")
print(f"GPU time: {inference_times_GPU.mean()} ({inference_times_GPU.std()})")
print(f"GPU to CPU time: {gputospu.mean()} ({gputospu.std()})")

print(f"Overall: {overall.mean()} ({overall.std()})")